# Node.js deployment export and application evaluation
Frontend: React + TypeScript. Backend: Node.js serverless functions. Hosting: Vercel Hobby.
ML training stays in Colab. The selected Random Forest is exported without retraining; predictions must match sklearn on every independent-test response before deployment.
Application source is edited in VS Code. Keep `deployment_source.zip` in the Drive project’s `deployment/` folder.

In [ ]:
import importlib.metadata as metadata
import subprocess, sys, os, signal
versions = {'numpy':'2.2.6','pandas':'2.2.3','scipy':'1.15.3','scikit-learn':'1.7.2','xgboost':'3.0.5','shap':'0.48.0','matplotlib':'3.10.3','joblib':'1.5.1'}
changed = False
for package, version in versions.items():
    try:
        changed |= metadata.version(package) != version
    except metadata.PackageNotFoundError:
        changed = True
if changed:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*[f'{p}=={v}' for p,v in versions.items()]])
    print('Environment installed. Runtime is restarting; run all cells again after reconnection.', flush=True)
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print('Pinned environment ready.')

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Final Research - C D Jayasinghe')

## 1. Export the selected model for Node.js

In [ ]:
import json, hashlib, subprocess, zipfile
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from google.colab import files

pipeline=joblib.load(ROOT/'models/h3_pipeline.joblib')
schema=json.loads((ROOT/'models/schema.json').read_text(encoding='utf-8'))
schema['reference_period']='past six months'
classifier=pipeline.named_steps['model']
if not isinstance(classifier,RandomForestClassifier) or list(pipeline.named_steps)!=['imputer','model']:
    raise ValueError('This deployment adapter requires the selected Random Forest pipeline. Adapt and revalidate the exporter if a future run selects another model.')
if list(classifier.classes_)!=[0,1,2]: raise ValueError('Unexpected class order.')

source_archive=ROOT/'deployment/deployment_source.zip'
if not source_archive.is_file(): raise FileNotFoundError('Place deployment_source.zip in deployment/.')
APP_ROOT=Path('/content/h3_node_application')
APP_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(source_archive) as archive:
    for member in archive.infolist():
        if not (APP_ROOT/member.filename).resolve().is_relative_to(APP_ROOT.resolve()):
            raise ValueError('Unsafe archive path.')
    archive.extractall(APP_ROOT)
MODEL_DIR=APP_ROOT/'deployment/models'
MODEL_DIR.mkdir(parents=True,exist_ok=True)
forest={'format':'sklearn-random-forest-v1','features':schema['features'],'classes':[0,1,2],
    'imputer':pipeline.named_steps['imputer'].statistics_.tolist(),
    'sklearn_version':schema['versions']['scikit-learn'],
    'pipeline_sha256':hashlib.sha256((ROOT/'models/h3_pipeline.joblib').read_bytes()).hexdigest(),
    'trees':[]}
for estimator in classifier.estimators_:
    tree=estimator.tree_
    forest['trees'].append({'left':tree.children_left.tolist(),'right':tree.children_right.tolist(),
        'feature':tree.feature.tolist(),'threshold':tree.threshold.tolist(),'value':tree.value[:,0,:].tolist()})
(MODEL_DIR/'h3_forest.json').write_text(json.dumps(forest,separators=(',',':'),allow_nan=False))
(MODEL_DIR/'schema.json').write_text(json.dumps(schema,indent=2,ensure_ascii=False),encoding='utf-8')
print('Exported',len(forest['trees']),'trees. No model retraining or change to selection.')

## 2. Verify Node/sklearn prediction equivalence

In [ ]:
# Reconstruct the exact analysis rows and frozen test partition from notebook 01.
audit=json.loads((ROOT/'outputs/data_audit.json').read_text())
sources=[p for p in (ROOT/'data').rglob('*.csv') if hashlib.sha256(p.read_bytes()).hexdigest()==audit['source_sha256']]
if len(sources)!=1: raise ValueError('Main-study source is missing or has changed.')
raw=pd.read_csv(sources[0],dtype='string').replace(r'^\s*$',pd.NA,regex=True).replace('Prefer not to answer',pd.NA)
raw.columns=raw.columns.str.strip()
consent=next(c for c in raw if c.startswith('Consent:'))
eligibility=next(c for c in raw if c.startswith('Do you meet all four eligibility criteria'))
study=raw.loc[raw[consent].eq('Yes, I agree').fillna(False)&raw[eligibility].eq('Yes, all four apply').fillna(False)].drop_duplicates()
def column(code):
    import re
    return next(c for c in study if re.match(rf'^{code}\s*[.:-]',c))
def numeric(series):
    return pd.to_numeric(series.str.strip().str.extract(r'^(\d+)(?:\s*[—–-]\s*.+)?$',expand=False),errors='coerce').astype(float)
X=pd.DataFrame({f:numeric(study[column(f)]) for f in schema['features']})
h3=numeric(study[column('H3')])
X=X.loc[h3.notna()&X.notna().any(axis=1)].reset_index(drop=True)
manifest=pd.read_csv(ROOT/'outputs/split_manifest.csv')
assert len(X)==len(manifest)==audit['analysis_n']
test=X.iloc[manifest.loc[manifest['partition'].eq('test'),'row']]

def cases(frame):
    expected=pipeline.predict_proba(frame)
    return [{'responses':[None if pd.isna(v) else float(v) for v in row], 'expected':p.tolist()}
            for row,p in zip(frame.to_numpy(),expected)]
private_fixture=APP_ROOT/'heldout_private.json'
private_fixture.write_text(json.dumps(cases(test),allow_nan=False))
rng=np.random.default_rng(42)
synthetic=rng.integers(0,5,size=(200,20)).astype(float)
synthetic[rng.random(synthetic.shape)<0.15]=np.nan
synthetic=np.vstack([np.repeat(np.arange(5)[:,None],20,axis=1),synthetic])
(MODEL_DIR/'parity_cases.json').write_text(json.dumps(cases(pd.DataFrame(synthetic,columns=schema['features'])),allow_nan=False))
check=subprocess.run(['node','tests/parity.mjs',str(private_fixture)],cwd=APP_ROOT,capture_output=True,text=True,check=True)
parity=json.loads(check.stdout)
(ROOT/'outputs/inference_parity.json').write_text(json.dumps(parity,indent=2))
display(pd.Series(parity))
assert parity['passed']

## 3. Functional, validation and explanation tests
Includes DVQ-R item mapping, exact and incomplete score arithmetic, observed-response support rules, and unchanged model inference.

In [ ]:
result=subprocess.run(['node','--test',*[str(p.relative_to(APP_ROOT)) for p in sorted((APP_ROOT/'tests').glob('*.test.mjs'))]],cwd=APP_ROOT,capture_output=True,text=True)
print(result.stdout)
(ROOT/'outputs/app_tests.txt').write_text(result.stdout,encoding='utf-8')
if result.returncode:
    print(result.stderr)
    raise RuntimeError('Resolve application test failures before deployment.')

## 4. Measure inference and explanation latency

In [ ]:
result=subprocess.run(['node','tests/benchmark.mjs'],cwd=APP_ROOT,capture_output=True,text=True,check=True)
latency=json.loads(result.stdout)
(ROOT/'outputs/app_latency_summary.json').write_text(json.dumps(latency,indent=2))
display(pd.Series(latency))

## 5. Export evaluated deployment artifacts
The bundle includes the Node model, schema, synthetic parity cases and aggregate evaluation results. The private test-response fixture and raw dataset are excluded.
Extract into the VS Code checkout, run the React production build and Node tests, and complete browser acceptance testing before deploying to Vercel.

In [ ]:
bundle=ROOT/'outputs/deployment_bundle.zip'
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in MODEL_DIR.glob('*.json'):
        archive.write(path,'deployment/models/'+path.name)
    names=['data_audit.json','split_summary.csv','cv_comparison.csv','test_comparison.csv',
        'model_selection.json','selected_model_confidence_intervals.csv','shap_global.csv',
        'app_tests.txt','inference_parity.json','app_latency_summary.json','environment.json']
    for name in names:
        path=ROOT/'outputs'/name
        if path.is_file(): archive.write(path,'evaluation/'+name)
    for path in (ROOT/'report_figures').glob('*.png'):
        archive.write(path,'evaluation/figures/'+path.name)
files.download(str(bundle))

## 6. Hosted verification
Import `ChamudiDJ/Final-Year-Research` into Vercel on the Hobby plan. Keep the project root at the repository root; `vercel.json` defines the React build and Node functions.
Complete the hosted browser acceptance checks, real user usability evaluation and domain-expert review in notebook 03.

In [ ]:
APP_URL='https://help-seeking-intention-dss.vercel.app'
import requests
from urllib.parse import urlparse
parsed=urlparse(APP_URL)
if parsed.scheme!='https' or not parsed.hostname or not parsed.hostname.endswith('.vercel.app'):
    raise ValueError('Use the deployed Vercel HTTPS URL.')
health=requests.get(APP_URL+'/api/health',timeout=60)
health.raise_for_status()
assert health.json()['status']=='ok'
catalog=requests.get(APP_URL+'/api/questionnaire',timeout=60)
catalog.raise_for_status()
assert catalog.json()['dss']['version']=='dvq-dss-v1'
support=requests.get(APP_URL+'/api/support',timeout=60)
support.raise_for_status()
assert support.json()['version']=='dvq-dss-v1'
synthetic={f'D{i:02d}':1 for i in range(1,21)}
def hosted_prediction(responses):
    response=requests.post(APP_URL+'/api/predict',json={'eligible':True,'responses':responses},timeout=60)
    response.raise_for_status()
    return response.json()
complete=hosted_prediction(synthetic)
assert complete['dvq']['score']==20
assert all(d['score']==4 for d in complete['dvq']['domains'])
partial=hosted_prediction({**synthetic,'D01':None})
assert partial['dvq']['score'] is None and partial['dvq']['observed_subtotal']==19
assert partial['dvq']['possible_range']==[19,23]
acceptance_path=ROOT/'outputs/dss_hosted_browser_acceptance.csv'
browser_status='pending'
if acceptance_path.is_file():
    acceptance=pd.read_csv(acceptance_path)
    if not acceptance.empty and {'case','passed','notes'}.issubset(acceptance.columns) and acceptance['passed'].eq(1).all():
        browser_status='passed'
check={'url':APP_URL,'api_status':'passed','dss_version':'dvq-dss-v1','browser_acceptance':browser_status,'participant_and_expert_evaluation':'pending real responses'}
(ROOT/'outputs/dss_hosting_check_colab.json').write_text(json.dumps(check,indent=2))
display(pd.Series(check))